# Flickr8k Dataset Exploration (Day 2)

**Goals:**
1. Number of images, captions per image, caption length distribution, vocabulary size
2. Visual inspection: sample images with captions
3. Data quality issues: duplicate captions, blank/short captions, corrupted images

**Expected folder structure (relative to this notebook):**
```
data/
  Images/           # all .jpg files
  captions.txt      # columns: image,caption  (Kaggle Flickr8k format)
notebooks/
  01_eda_flickr8k.ipynb   <- this file
```
If you have the older `Flickr8k.token.txt` format instead, see the note in the loading cell below.

In [1]:
import os
import re
import random
from collections import Counter

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image, UnidentifiedImageError

DATA_DIR = "../data"
IMAGES_DIR = os.path.join(DATA_DIR, "Images")
CAPTIONS_FILE = os.path.join(DATA_DIR, "captions.txt")

pd.set_option('display.max_colwidth', 100)

## 1. Load captions

Handles the common Kaggle `captions.txt` (image,caption per line, header row).
If you instead have `Flickr8k.token.txt` (format: `image.jpg#0\tcaption text`),
uncomment the alternate loader below.

In [2]:
# --- Standard captions.txt loader ---
df = pd.read_csv(CAPTIONS_FILE)
df.columns = [c.strip().lower() for c in df.columns]  # normalize e.g. 'image','caption'
assert {'image', 'caption'}.issubset(df.columns), f"Unexpected columns: {df.columns.tolist()}"

# --- Alternate loader for Flickr8k.token.txt format ---
# rows = []
# with open(os.path.join(DATA_DIR, 'Flickr8k.token.txt'), 'r', encoding='utf-8') as f:
#     for line in f:
#         line = line.strip()
#         if not line:
#             continue
#         img_tag, caption = line.split('\t')
#         image = img_tag.split('#')[0]
#         rows.append((image, caption))
# df = pd.DataFrame(rows, columns=['image', 'caption'])

print(df.shape)
df.head(10)

ParserError: Error tokenizing data. C error: Expected 1 fields in line 37, saw 2


## 2. Basic statistics

In [3]:
num_captions = len(df)
num_unique_images = df['image'].nunique()
captions_per_image = df.groupby('image').size()

stats = {
    'Total captions': num_captions,
    'Unique images (in captions file)': num_unique_images,
    'Min captions/image': captions_per_image.min(),
    'Max captions/image': captions_per_image.max(),
    'Mean captions/image': round(captions_per_image.mean(), 2),
}

# Actual image files on disk
if os.path.isdir(IMAGES_DIR):
    image_files = [f for f in os.listdir(IMAGES_DIR) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    stats['Image files found on disk'] = len(image_files)
else:
    image_files = []
    stats['Image files found on disk'] = 'IMAGES_DIR not found'

stats_df = pd.DataFrame(list(stats.items()), columns=['Metric', 'Value'])
stats_df

NameError: name 'df' is not defined

In [ ]:
# Captions-per-image distribution table
cpi_counts = captions_per_image.value_counts().sort_index()
cpi_table = cpi_counts.rename_axis('captions_per_image').reset_index(name='num_images')
cpi_table

## 3. Caption length distribution

In [ ]:
def tokenize(text):
    return re.findall(r"[a-zA-Z']+", str(text).lower())

df['tokens'] = df['caption'].apply(tokenize)
df['caption_len'] = df['tokens'].apply(len)

length_stats = df['caption_len'].describe()
print(length_stats)

plt.figure(figsize=(8, 5))
plt.hist(df['caption_len'], bins=range(0, df['caption_len'].max() + 2), edgecolor='black')
plt.title('Caption Length Distribution (words per caption)')
plt.xlabel('Number of words')
plt.ylabel('Number of captions')
plt.tight_layout()
plt.savefig('caption_length_histogram.png', dpi=150)
plt.show()

## 4. Vocabulary size

In [ ]:
all_tokens = [tok for toks in df['tokens'] for tok in toks]
vocab_counter = Counter(all_tokens)
vocab_size = len(vocab_counter)

print(f"Total word occurrences: {len(all_tokens)}")
print(f"Vocabulary size (unique words): {vocab_size}")

top_words = pd.DataFrame(vocab_counter.most_common(20), columns=['word', 'count'])
top_words

In [ ]:
plt.figure(figsize=(8, 6))
plt.barh(top_words['word'][::-1], top_words['count'][::-1])
plt.title('Top 20 Most Frequent Words')
plt.xlabel('Count')
plt.tight_layout()
plt.savefig('top_words.png', dpi=150)
plt.show()

## 5. Visual inspection: sample image-caption grid

In [ ]:
def show_sample_grid(n=6, seed=42):
    if not image_files:
        print("No images found on disk — skipping visual inspection.")
        return
    random.seed(seed)
    sample_images = random.sample(list(df['image'].unique()), min(n, df['image'].nunique()))

    cols = 3
    rows = (len(sample_images) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(15, rows * 5))
    axes = axes.flatten() if rows > 1 else [axes] if cols == 1 else axes

    for ax, img_name in zip(axes, sample_images):
        img_path = os.path.join(IMAGES_DIR, img_name)
        caption = df[df['image'] == img_name]['caption'].iloc[0]
        try:
            img = Image.open(img_path)
            ax.imshow(img)
        except Exception as e:
            ax.text(0.5, 0.5, f"Could not load\n{img_name}", ha='center', va='center')
        ax.set_title(caption, fontsize=9, wrap=True)
        ax.axis('off')

    for ax in axes[len(sample_images):]:
        ax.axis('off')

    plt.tight_layout()
    plt.savefig('sample_image_caption_grid.png', dpi=150)
    plt.show()

show_sample_grid(n=6)

## 6. Data quality checks

### 6a. Duplicate captions

In [ ]:
# Exact duplicate captions (same text repeated anywhere in dataset)
dup_mask = df.duplicated(subset=['caption'], keep=False)
duplicate_captions_df = df[dup_mask].sort_values('caption')
print(f"Total duplicate caption rows: {dup_mask.sum()} ({dup_mask.mean()*100:.2f}% of all captions)")
duplicate_captions_df.head(10)

In [ ]:
# Duplicate captions WITHIN the same image (a stricter/more meaningful issue)
dup_within_image = df.groupby('image')['caption'].apply(lambda x: x.duplicated().any())
images_with_dup_captions = dup_within_image[dup_within_image].index.tolist()
print(f"Images that have duplicate captions among their own 5: {len(images_with_dup_captions)}")
images_with_dup_captions[:10]

### 6b. Blank / very short captions

In [ ]:
MIN_WORDS = 2  # threshold: captions with fewer words are flagged

blank_captions = df[df['caption'].astype(str).str.strip() == '']
short_captions = df[(df['caption_len'] < MIN_WORDS) & (df['caption'].astype(str).str.strip() != '')]

print(f"Blank captions: {len(blank_captions)}")
print(f"Short captions (< {MIN_WORDS} words): {len(short_captions)}")
short_captions.head(10)

### 6c. Corrupted / unreadable images

In [ ]:
def find_corrupted_images(image_dir, files):
    corrupted = []
    for fname in files:
        path = os.path.join(image_dir, fname)
        try:
            with Image.open(path) as img:
                img.verify()  # checks integrity without full decode
        except (UnidentifiedImageError, OSError, Exception) as e:
            corrupted.append((fname, str(e)))
    return corrupted

if image_files:
    corrupted_images = find_corrupted_images(IMAGES_DIR, image_files)
    print(f"Corrupted/unreadable images found: {len(corrupted_images)} out of {len(image_files)}")
    corrupted_df = pd.DataFrame(corrupted_images, columns=['image', 'error'])
else:
    corrupted_df = pd.DataFrame(columns=['image', 'error'])
    print("No images found on disk to check.")

corrupted_df

In [ ]:
# Captions referencing images that don't actually exist on disk
if image_files:
    missing_images = sorted(set(df['image'].unique()) - set(image_files))
    print(f"Captions pointing to missing image files: {len(missing_images)}")
else:
    missing_images = []

## 7. Summary report

In [ ]:
summary = {
    'Total images (files on disk)': len(image_files),
    'Total unique images in captions.txt': num_unique_images,
    'Total captions': num_captions,
    'Mean captions/image': round(captions_per_image.mean(), 2),
    'Vocabulary size': vocab_size,
    'Mean caption length (words)': round(df['caption_len'].mean(), 2),
    'Max caption length (words)': df['caption_len'].max(),
    'Duplicate caption rows (global)': int(dup_mask.sum()),
    'Images with internal duplicate captions': len(images_with_dup_captions),
    'Blank captions': len(blank_captions),
    f'Short captions (< {MIN_WORDS} words)': len(short_captions),
    'Corrupted images': len(corrupted_df),
    'Captions with missing image file': len(missing_images),
}

summary_df = pd.DataFrame(list(summary.items()), columns=['Metric', 'Value'])
summary_df.to_csv('eda_summary.csv', index=False)
summary_df

## Notes / Next steps
- Adjust `MIN_WORDS` threshold if a different definition of "short" caption is preferred.
- If duplicate/short/corrupted counts are high, decide with mentor whether to clean before modeling (Day 3+).
- Outputs saved alongside this notebook: `caption_length_histogram.png`, `top_words.png`, `sample_image_caption_grid.png`, `eda_summary.csv`.